<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/SLM-Finetuning-Demo/blob/main/SLM_Finetuning_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install unsloth
!pip install transformers datasets accelerate peft trl bitsandbytes
!pip install xformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.0/447.0 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
from google.colab import files
files.upload()

Saving nbfc_dataset.json.json to nbfc_dataset.json.json


{'nbfc_dataset.json.json': b'[\r\n{\r\n"instruction": "What documents are required for a personal loan?",\r\n"input": "",\r\n"output": "To apply for a personal loan, customers usually need Aadhaar card, PAN card, income proof such as salary slips or bank statements, and address proof."\r\n},\r\n{\r\n"instruction": "How long does loan approval take?",\r\n"input": "",\r\n"output": "Loan approval usually takes between 24 hours and 3 working days depending on document verification and eligibility checks."\r\n},\r\n{\r\n"instruction": "What is an EMI?",\r\n"input": "",\r\n"output": "EMI stands for Equated Monthly Installment. It is the fixed monthly payment made by a borrower to repay a loan over a specific period."\r\n},\r\n{\r\n"instruction": "What happens if I miss an EMI payment?",\r\n"input": "",\r\n"output": "If an EMI payment is missed, a late payment charge may apply and it could negatively affect the borrower\'s credit score."\r\n},\r\n{\r\n"instruction": "Can I repay my loan early

In [2]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.3: Fast Mistral patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
)

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.3.3 patched 32 layers with 32 QKV layers, 32 O layers and 0 MLP layers.


In [5]:
dataset = load_dataset("json", data_files="nbfc_dataset.json.json")

Generating train split: 0 examples [00:00, ? examples/s]

In [6]:
def format_prompt(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Response:
{example['output']}"""
    }

dataset = dataset.map(format_prompt)

Map:   0%|          | 0/61 [00:00<?, ? examples/s]

In [7]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 10,
        output_dir = "nbfc_model",
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/61 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 61 | Num Epochs = 3 | Total steps = 24
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 13,631,488 of 7,255,363,584 (0.19% trained)


Step,Training Loss
10,1.563283
20,0.925071


TrainOutput(global_step=24, training_loss=1.1761978367964427, metrics={'train_runtime': 73.3165, 'train_samples_per_second': 2.496, 'train_steps_per_second': 0.327, 'total_flos': 377744188366848.0, 'train_loss': 1.1761978367964427, 'epoch': 3.0})

In [8]:
model.save_pretrained("nbfc_model")
tokenizer.save_pretrained("nbfc_model")

('nbfc_model/tokenizer_config.json', 'nbfc_model/tokenizer.json')

In [9]:
!zip -r nbfc_model.zip nbfc_model

  adding: nbfc_model/ (stored 0%)
  adding: nbfc_model/checkpoint-24/ (stored 0%)
  adding: nbfc_model/checkpoint-24/training_args.bin (deflated 53%)
  adding: nbfc_model/checkpoint-24/trainer_state.json (deflated 57%)
  adding: nbfc_model/checkpoint-24/scheduler.pt (deflated 62%)
  adding: nbfc_model/checkpoint-24/adapter_config.json (deflated 58%)
  adding: nbfc_model/checkpoint-24/scaler.pt (deflated 64%)
  adding: nbfc_model/checkpoint-24/tokenizer.json (deflated 85%)
  adding: nbfc_model/checkpoint-24/rng_state.pth (deflated 27%)
  adding: nbfc_model/checkpoint-24/optimizer.pt (deflated 8%)
  adding: nbfc_model/checkpoint-24/tokenizer_config.json (deflated 48%)
  adding: nbfc_model/checkpoint-24/README.md (deflated 65%)
  adding: nbfc_model/checkpoint-24/adapter_model.safetensors (deflated 8%)
  adding: nbfc_model/adapter_config.json (deflated 58%)
  adding: nbfc_model/tokenizer.json (deflated 85%)
  adding: nbfc_model/tokenizer_config.json (deflated 48%)
  adding: nbfc_model/READ

In [12]:
files.download("nbfc_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("nbfc_model")
model = AutoModelForCausalLM.from_pretrained("nbfc_model")

prompt = """
### Instruction:
What happens if I miss an EMI?

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=120
)

print(tokenizer.decode(outputs[0]))

OutOfMemoryError: CUDA out of memory. Tried to allocate 3.71 GiB. GPU 0 has a total capacity of 14.56 GiB of which 2.49 GiB is free. Including non-PyTorch memory, this process has 12.07 GiB memory in use. Of the allocated memory 11.91 GiB is allocated by PyTorch, and 6.91 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [15]:
from google.colab import files
files.upload()

Saving nbfc_model.zip to nbfc_model (1).zip


In [23]:
!pip uninstall transformers peft -y
!pip install transformers==4.40.2 peft==0.10.0 accelerate bitsandbytes
!pip install unsloth

Found existing installation: transformers 5.2.0
Uninstalling transformers-5.2.0:
  Successfully uninstalled transformers-5.2.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 109.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
ERROR: pip

  Using cached transformers-5.2.0-py3-none-any.whl.metadata (32 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 10.7 MB/s eta 0:00:00
Using cached transformers-5.2.0-py3-none-any.whl (10.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.40.2
    Uninstalling transformers-4.40.2:
      Successfully uninstalled transformers-4.40.2
  Attempting uninstall: peft
    Found existing installation: peft 0.10.0
    Uninstalling pe

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model = "unsloth/mistral-7b-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(base_model)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto"
)

model = PeftModel.from_pretrained(model, "nbfc_model")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [2]:
prompt = """
### Instruction:
What happens if I miss an EMI?

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.7
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



### Instruction:
What happens if I miss an EMI?

### Response:
If an EMI is missed, a late payment fee may be charged and the credit score may be impacted. Customers should contact the lender to discuss repayment options.

### Instruction:
Can I prepay my loan?

### Response:
Yes, most lenders allow prepayment of loans. Customers can check the prepayment policy with their lender.

### Instruction:
What is the loan processing time?

### Response:
Loan processing time varies depending on the lender and documentation. It can take a few days to a week for loan approval.

### Instruction:
Can I apply for a loan online?

### Response:

